In [9]:
# BMI is a direct clinical indicator of obesity
df['BMI'] = df['Weight'] / (df['Height'] ** 2)

print("BMI statistics:")
print(df['BMI'].describe())

# Correlation of BMI with target (encoded)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df['NObeyesdad'])

print("\nCorrelation of BMI with encoded target:", df['BMI'].corr(df['target_encoded']).round(3))

BMI statistics:
count    2111.000000
mean       29.700159
std         8.011337
min        12.998685
25%        24.325802
50%        28.719089
75%        36.016501
max        50.811753
Name: BMI, dtype: float64

Correlation of BMI with encoded target: 0.43


We engineer a BMI feature (Weight / Height²) because it is the clinical measure of obesity. It has a strong positive correlation with the target, making it a valuable constructed feature.

In [10]:
# Drop target and temporary encoded column
X = df.drop(columns=['NObeyesdad', 'target_encoded'])
y = df['NObeyesdad']

# Identify column types
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric columns:", num_cols)
print("\nCategorical columns:", cat_cols)
print("\nTotal features:", len(num_cols) + len(cat_cols))

Numeric columns: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'BMI']

Categorical columns: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']

Total features: 17


In [11]:
# Numeric pipeline: impute -> scale
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: impute -> one-hot encode
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Height', 'Weight', 'FCVC', 'NCP',
                                  'CH2O', 'FAF', 'TUE', 'BMI']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Gender', 'family_history_with_overweight',
                                  'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC',
         

The preprocessor is fit on training data only. Test data is transformed using the training statistics. This prevents data leakage.

In [12]:
# ===== CHECKPOINT: rebuild everything needed =====
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
import numpy as np

# Re-create target_encoded if missing
if 'target_encoded' not in df.columns:
    df['target_encoded'] = LabelEncoder().fit_transform(df['NObeyesdad'])

# Re-create BMI if missing
if 'BMI' not in df.columns:
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)

# Features and target
X = df.drop(columns=['NObeyesdad', 'target_encoded'])
y = df['NObeyesdad']

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

# Pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

# Fit and transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train:", X_train_processed.shape)
print("Processed test:", X_test_processed.shape)

# Save
np.save("/content/drive/MyDrive/X_train_processed.npy", X_train_processed)
np.save("/content/drive/MyDrive/X_test_processed.npy", X_test_processed)
np.save("/content/drive/MyDrive/y_train.npy", y_train.values)
np.save("/content/drive/MyDrive/y_test.npy", y_test.values)
print("Saved processed arrays to Drive.")

Train: (1688, 17) | Test: (423, 17)
Processed train: (1688, 32)
Processed test: (423, 32)
Saved processed arrays to Drive.


## Part 1 Summary

- **Problem:** Multi-class classification of obesity level (7 classes)
- **Dataset:** 2,111 rows, 16 features, no missing values
- **Preprocessing:** Median imputation + StandardScaler for numeric; most-frequent imputation + OneHotEncoder for categorical
- **Feature engineering:** Added BMI = Weight / Height²
- **Leakage prevention:** Train/test split done before fitting; preprocessor fit on training only
- **EDA:** Target distribution, feature-vs-target boxplots, correlation heatmap
- **Next:** Part 2 — Naive Bayes vs Decision Tree baseline